In [49]:
import sys
from collections import Counter
from pathlib import Path

import spacy
import unicodedataplus as ud
from symspellpy import SymSpell, Verbosity

In [12]:
def scriptScore(token, targetScript: str = 'ARABIC'):
    """
    Returns a score from 0.0 to 1.0 representing the proportion
    of characters in 'token' belonging to 'targetScript'.
    """
    if not token:
        return 0.0

    matchCount = 0
    # Normalize target script to uppercase for comparison
    targetScript = targetScript.upper()

    ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"
    for ch in token:
        script = ud.script(ch).upper() if ch not in ArabicVowels else 'ARABIC'
        if script in (targetScript, "INHERITED"):
            matchCount += 1

    return matchCount / len(token)

In [13]:
ArabicVowels = "\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652"

for ch in ArabicVowels:
    print(f"'{ch}': {scriptScore(ch)} - {ud.script(ch).upper()}")

'ً': 1.0 - INHERITED
'ٌ': 1.0 - INHERITED
'ٍ': 1.0 - INHERITED
'َ': 1.0 - INHERITED
'ُ': 1.0 - INHERITED
'ِ': 1.0 - INHERITED
'ّ': 1.0 - INHERITED
'ْ': 1.0 - INHERITED


In [5]:
ud.script(" ")

'Common'

In [6]:
def tokenize(text: str):
    nlp = spacy.blank('ur')

    # Spacy based tokenization
    doc = nlp(text)
    return  doc

In [7]:
def tokens2Vocab(doc, isAlpha: bool = True, scoreThreshold: float = 0.9) -> tuple[Counter, int]:

    words = [token.text for token in doc if (isAlpha and token.is_alpha) and scriptScore(token.text) >= scoreThreshold]
    wordCount = len(words)

    vocab = Counter(words)

    return vocab, wordCount

In [8]:
urduText = """
فرض کرو ہم تارے ہوتے
ايک دوجے کو دور دور سے ديکھ ديکھ کر جلتے بجتے
اور پھر ايک دن
شاخِ فلک سے گرتے اور تاريک خلاؤں ميں کھو جاتے
دريا کے دو دھارے ہوتے،
اپنى اپنى موج ميں بہتے
اور سمندر تک اس اَندھى، وحشى اور منہ زور مسافت
کے جادو ميں تنہا رہتے
فرض کررو ہم بھور سمے کے پنچھى ہوتے،
اُڑتے اُڑتے ايک دوجے کو چھوتے اور پھر
کھلے گگن کى گہرى اور بے صرفہ آنکھوں ميں کھو جاتے
اور بہار کے جھونکے ہوتے،
موسم کے اک بےنقشہ خواب ميں ملتے
ملتے اور جدا ہو جاتے
خشک زمينوں کے ہاتھوں پر سبز لکيريں کندہ کرتے
اور ان ديکھے سپنے بوتے
اپنے اپنے رو کر چين سے سو جاتے
فرض کرو ہم جو کچھ اب ہيں وہ ناں ہوتے۔۔۔
"""

In [8]:
txt = Path(r"training_Urdu_PK.txt").read_text(encoding='utf8')
if len(txt) > 1_000_000:
    txt = txt[:1_000_000]
    print("Trimming input text size to 1,000,000")
else:
    print(f"Input text size: {len(txt):,}")

Input text size: 119,238


In [9]:
doc = tokenize(txt)
tokenCount = len(doc)
vocabulary, wordCount = tokens2Vocab(doc)
vocabCount = len(vocabulary)
print(f"Totals:: {tokenCount=:,} {wordCount=:,} {vocabCount=:,}")


Totals:: tokenCount=28,639 wordCount=25,295 vocabCount=4,041


In [10]:
print(f"{'Word':<15} | Frequency | Score")
print("-" * 25)
for w, f in vocabulary.most_common():
    s = scriptScore(w)
    if s < 1.0 :
        print(f"{w:15} | {f:9} | {s}")

print(f"\n{'Word':<15} | Frequency n(%) | Score")
print("-" * 25)
for w, f in vocabulary.most_common(n=100):
    s = scriptScore(w)
    print(f"{w:15} | {f:9}({(f/wordCount)*100:0.2f}%) | {s}")

Word            | Frequency | Score
-------------------------

Word            | Frequency n(%) | Score
-------------------------
کے              |       989(3.91%) | 1.0
کی              |       841(3.32%) | 1.0
ہے              |       829(3.28%) | 1.0
میں             |       800(3.16%) | 1.0
اور             |       728(2.88%) | 1.0
کا              |       548(2.17%) | 1.0
سے              |       547(2.16%) | 1.0
ہیں             |       384(1.52%) | 1.0
اس              |       363(1.44%) | 1.0
کو              |       285(1.13%) | 1.0
نے              |       280(1.11%) | 1.0
ان              |       262(1.04%) | 1.0
بھی             |       228(0.90%) | 1.0
کہ              |       197(0.78%) | 1.0
ایک             |       187(0.74%) | 1.0
کر              |       169(0.67%) | 1.0
پر              |       169(0.67%) | 1.0
و               |       167(0.66%) | 1.0
یہ              |       165(0.65%) | 1.0
نہیں            |       165(0.65%) | 1.0
کیا             |       163(0.64%) | 1.0
وہ       

In [26]:
Words5kTxt = Path(r"Urdu5k.txt").read_text(encoding='utf8')
lines = [line.strip() for line in Words5kTxt.splitlines()]
len(lines)

10000

In [24]:
vocab = []
i = 0
while i < len(lines):
    l1 = scriptScore(lines[i]) == 1.0
    l2 = scriptScore(lines[i+1]) == 1.0
    if l1 and l2:
        w = lines[i]
        f = lines[i+1]
        assert f.isdigit(), f"{i+1} line is not freq"
        vocab.append((w, f))
    else:
        print(lines[i], lines[i+1])
    i += 2

In [25]:
len(vocab)

5000

In [41]:
for i, l in enumerate(lines, start=1):
    if i % 2 == 0:
        # print(i, l)
        assert l.strip().isdigit(), f"{i} line is not numeric"
    else:
        assert scriptScore(l.strip()) == 1.0

In [31]:
lines[:]

['ﮐﮯ',
 '743949',
 'ﻣﻴﮟ',
 '582882',
 'ﮐﯽ',
 '575545',
 'ﮨﮯ',
 '466908',
 'اور',
 '413788']

In [80]:
CORRECT_URDU_CHARACTERS: dict = {'آ': ['ﺁ', 'ﺂ'],
                                 'أ': ['ﺃ'],
                                 'ا': ['ﺍ', 'ﺎ', ],
                                 'ب': ['ﺏ', 'ﺐ', 'ﺑ', 'ﺒ'],
                                 'پ': ['ﭖ', 'ﭗ', 'ﭘ', 'ﭙ'],
                                 'ت': ['ﺕ', 'ﺖ', 'ﺗ', 'ﺘ'],
                                 'ٹ': ['ﭦ', 'ﭧ', 'ﭨ', 'ﭩ'],
                                 'ث': ['ﺛ', 'ﺜ', 'ﺚ'],
                                 'ج': ['ﺝ', 'ﺞ', 'ﺟ', 'ﺠ'],
                                 'ح': ['ﺡ', 'ﺣ', 'ﺤ', 'ﺢ'],
                                 'خ': ['ﺧ', 'ﺨ', 'ﺦ'],
                                 'د': ['ﺩ', 'ﺪ'],
                                 'ذ': ['ﺬ', 'ﺫ'],
                                 'ر': ['ﺭ', 'ﺮ'],
                                 'ز': ['ﺯ', 'ﺰ', ],
                                 'س': ['ﺱ', 'ﺲ', 'ﺳ', 'ﺴ', ],
                                 'ش': ['ﺵ', 'ﺶ', 'ﺷ', 'ﺸ'],
                                 'ص': ['ﺹ', 'ﺺ', 'ﺻ', 'ﺼ', ],
                                 'ض': ['ﺽ', 'ﺾ', 'ﺿ', 'ﻀ'],
                                 'ط': ['ﻃ', 'ﻄ', 'ﻂ'],
                                 'ظ': ['ﻅ', 'ﻇ', 'ﻆ', 'ﻈ'],
                                 'ع': ['ﻉ', 'ﻊ', 'ﻋ', 'ﻌ', ],
                                 'غ': ['ﻍ', 'ﻏ', 'ﻐ', 'ﻎ'],
                                 'ف': ['ﻑ', 'ﻒ', 'ﻓ', 'ﻔ', ],
                                 'ق': ['ﻕ', 'ﻖ', 'ﻗ', 'ﻘ', ],
                                 'ل': ['ﻝ', 'ﻞ', 'ﻟ', 'ﻠ', ],
                                 'م': ['ﻡ', 'ﻢ', 'ﻣ', 'ﻤ', ],
                                 'ن': ['ﻥ', 'ﻦ', 'ﻧ', 'ﻨ', ],
                                 'چ': ['ﭺ', 'ﭻ', 'ﭼ', 'ﭽ'],
                                 'ڈ': ['ﮈ', 'ﮉ'],
                                 'ڑ': ['ﮍ', 'ﮌ'],
                                 'ژ': ['ﮋ', ],
                                 'ک': ['ﮎ', 'ﮏ', 'ﮐ', 'ﮑ', 'ﻛ', 'ك'],
                                 'گ': ['ﮒ', 'ﮓ', 'ﮔ', 'ﮕ'],
                                 'ں': ['ﮞ', 'ﮟ'],
                                 'و': ['ﻮ', 'ﻭ', 'ﻮ', ],
                                 'ؤ': ['ﺅ'],
                                 'ھ': ['ﮪ', 'ﮬ', 'ﮭ', 'ﻬ', 'ﻫ', 'ﮫ'],
                                 'ہ': ['ﻩ', 'ﮦ', 'ﻪ', 'ﮧ', 'ﮩ', 'ﮨ', 'ه', ],
                                 'ۂ': [],
                                 'ۃ': ['ة'],
                                 'ء': ['ﺀ'],
                                 'ی': ['ﯼ', 'ى', 'ﯽ', 'ﻰ', 'ﻱ', 'ﻲ', 'ﯾ', 'ﯿ', 'ي', 'ﻳ', 'ﻴ'],
                                 'ئ': ['ﺋ', 'ﺌ', ],
                                 'ے': ['ﮮ', 'ﮯ' ],
                                 'ۓ': [],
                                 '۰': ['٠'],
                                 '۱': ['١'],
                                 '۲': ['٢'],
                                 '۳': ['٣'],
                                 '۴': ['٤'],
                                 '۵': ['٥'],
                                 '۶': ['٦'],
                                 '۷': ['٧'],
                                 '۸': ['٨'],
                                 '۹': ['٩'],
                                 '۔': [],
                                 '؟': [],
                                 '٫': [],
                                 '،': [],
                                 'لا': ['ﻻ', 'ﻼ', 'ﻵ'],
                                 '': ['ـ']

                                 }

_TRANSLATOR = {}
for key, value in CORRECT_URDU_CHARACTERS.items():
    _TRANSLATOR.update(dict.fromkeys(map(ord, value), key))

In [63]:
def getUniRep(txt):
    assert txt is not None
    assert type(txt) == str

    uc = [f"U+{ord(c):04X}" for c in txt]
    return " ".join(uc[::-1])

In [10]:
def isArabicBlock(text):
    """Checks if all characters are within the main Arabic Unicode block."""
    return all('\u0600' <= char <= '\u06FF' for char in text if not char.isspace())

In [82]:
with open(r"Urdu5k.sym", "w", encoding="utf-8") as dic:
    for i, (w, f) in enumerate(vocab):
        ww = w.translate(_TRANSLATOR)
        if not isArabicBlock(ww):
            print(i, getUniRep(w), w)
            ww = w.translate(_TRANSLATOR)
            print(i, getUniRep(ww), ww, '\n')
        dic.write(f"{ww}${f}\n")

In [68]:
w, f = vocab[1427]
print(getUniRep(w), w)
ww = w.translate(_TRANSLATOR)
print(getUniRep(ww), ww)

U+06BA U+FEEE U+FEE4 U+FEF4 U+FEC8 U+FEE8 U+FE97 ﺗﻨﻈﻴﻤﻮں
U+06BA U+0648 U+0645 U+06CC U+0638 U+0646 U+062A تنظیموں


In [1]:
def normalize2Urdu(token: str) -> str:
    """
    Standardizes Urdu tokens by:
    1. Mapping positional variants (Initial/Medial/Final forms) to base characters.
    2. Converting Arabic/Persian range characters to Urdu standard block.
    3. Removing diacritics and non-spacing marks.
    """
    if not token:
        return token

    # 1. Compatibility Decomposition (NFKC)
    # This automatically converts most positional variants (e.g., ﻒ, ﻘ, ﻂ)
    # from the Presentation Forms blocks to their standard Arabic script bases.
    token = ud.normalize('NFKC', token)

    # 2. Urdu-Specific Base Character Mapping
    # After NFKC, some chars might be in the 'Arabic' block (0643).
    # We must force them into the 'Urdu' preferred block.
    urduBaseMapping = {
        # Kaf variants
        '\u0643': '\u06a9', # Arabic Kaf -> Urdu Kaf
        '\u06a8': '\u06a9', # Swash Kaf -> Urdu Kaf

        # Yeh variants
        '\u064a': '\u06cc', # Arabic Yeh -> Urdu Chooti Yeh
        '\u0649': '\u06cc', # Alif Maqsura -> Urdu Chooti Yeh
        '\u06d2': '\u06d2', # Preserve Bari Yeh

        # Heh variants
        '\u0647': '\u06c1', # Arabic Heh -> Urdu Gol Heh
        '\u0629': '\u06c1', # Ta Marbuta -> Urdu Gol Heh

        # Zero-Width non-joiners (often used in positional variants)
        '\u200c': '',
    }

    for target, replacement in urduBaseMapping.items():
        token = token.replace(target, replacement)

    # 3. Strip Diacritics (Zabar, Zer, Pesh, etc.)
    # We use NFD to isolate marks, then filter them out.
    nfd_form = ud.normalize('NFD', token)
    token = "".join([c for c in nfd_form if not ud.combining(c)])

    return ud.normalize('NFC', token)

In [11]:
# Example: Testing with a 'Medial' form variant
# Character 'ﻒ' (Final Fe) becomes 'ف'
testToken = "\ufeef\u064e" # Final Fe + Zabar
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalize2Urdu(testToken)
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ﻯَ [U+064E U+FEEF]
Normalized: ی [U+06CC]


In [4]:
URDU_VARIANT_MAP = {
    # ALIF (ا)
    '\u0627': [
        '\u0627',  # ARABIC LETTER ALIF
        '\uFE8D',  # ARABIC LETTER ALIF ISOLATED FORM
        '\uFE8E',  # ARABIC LETTER ALIF FINAL FORM
    ],
    # ALEF MADDA (آ)
    '\u0622': [
        '\u0622',  # ARABIC LETTER ALEF WITH MADDA ABOVE
        '\uFE81',  # ALEF WITH MADDA ABOVE ISOLATED FORM
        '\uFE82',  # ALEF WITH MADDA ABOVE FINAL FORM
    ],
    # BE (ب)
    '\u0628': [
        '\u0628',  # ARABIC LETTER BE
        '\uFE8F',  # ARABIC LETTER BE ISOLATED FORM
        '\uFE90',  # ARABIC LETTER BE FINAL FORM
        '\uFE91',  # ARABIC LETTER BE INITIAL FORM
        '\uFE92',  # ARABIC LETTER BE MEDIAL FORM
    ],
    # PE (پ)
    '\u067E': [
        '\u067E',  # ARABIC LETTER PE
        '\uFB56',  # ARABIC LETTER PE ISOLATED FORM
        '\uFB57',  # ARABIC LETTER PE FINAL FORM
        '\uFB58',  # ARABIC LETTER PE INITIAL FORM
        '\uFB59',  # ARABIC LETTER PE MEDIAL FORM
    ],
    # TE (ت)
    '\u062A': [
        '\u062A',  # ARABIC LETTER TE
        '\uFE95',  # ARABIC LETTER TE ISOLATED FORM
        '\uFE96',  # ARABIC LETTER TE FINAL FORM
        '\uFE97',  # ARABIC LETTER TE INITIAL FORM
        '\uFE98',  # ARABIC LETTER TE MEDIAL FORM
    ],
    # TTE (ٹ)
    '\u0679': [
        '\u0679',  # ARABIC LETTER TTE
        '\uFB66',  # ARABIC LETTER TTE ISOLATED FORM
        '\uFB67',  # ARABIC LETTER TTE FINAL FORM
        '\uFB68',  # ARABIC LETTER TTE INITIAL FORM
        '\uFB69',  # ARABIC LETTER TTE MEDIAL FORM
    ],
    # SE (ث)
    '\u062B': [
        '\u062B',  # ARABIC LETTER THE
        '\uFE99',  # ARABIC LETTER THE ISOLATED FORM
        '\uFE9A',  # ARABIC LETTER THE FINAL FORM
        '\uFE9B',  # ARABIC LETTER THE INITIAL FORM
        '\uFE9C',  # ARABIC LETTER THE MEDIAL FORM
    ],
    # JEEM (ج)
    '\u062C': [
        '\u062C',  # ARABIC LETTER JEEM
        '\uFE9D',  # ARABIC LETTER JEEM ISOLATED FORM
        '\uFE9E',  # ARABIC LETTER JEEM FINAL FORM
        '\uFE9F',  # ARABIC LETTER JEEM INITIAL FORM
        '\uFEA0',  # ARABIC LETTER JEEM MEDIAL FORM
    ],
    # CHE (چ)
    '\u0686': [
        '\u0686',  # ARABIC LETTER CHE
        '\uFB7A',  # ARABIC LETTER CHE ISOLATED FORM
        '\uFB7B',  # ARABIC LETTER CHE FINAL FORM
        '\uFB7C',  # ARABIC LETTER CHE INITIAL FORM
        '\uFB7D',  # ARABIC LETTER CHE MEDIAL FORM
    ],
    # BARI HE (ح)
    '\u062D': [
        '\u062D',  # ARABIC LETTER HAA
        '\uFEA1',  # ARABIC LETTER HAA ISOLATED FORM
        '\uFEA2',  # ARABIC LETTER HAA FINAL FORM
        '\uFEA3',  # ARABIC LETTER HAA INITIAL FORM
        '\uFEA4',  # ARABIC LETTER HAA MEDIAL FORM
    ],
    # KHE (خ)
    '\u062E': [
        '\u062E',  # ARABIC LETTER KHAA
        '\uFEA5',  # ARABIC LETTER KHAA ISOLATED FORM
        '\uFEA6',  # ARABIC LETTER KHAA FINAL FORM
        '\uFEA7',  # ARABIC LETTER KHAA INITIAL FORM
        '\uFEA8',  # ARABIC LETTER KHAA MEDIAL FORM
    ],
    # DAL (د)
    '\u062F': [
        '\u062F',  # ARABIC LETTER DAL
        '\uFEA9',  # ARABIC LETTER DAL ISOLATED FORM
        '\uFEAA',  # ARABIC LETTER DAL FINAL FORM
    ],
    # DDAL (ڈ)
    '\u0688': [
        '\u0688',  # ARABIC LETTER DDAL
        '\uFB88',  # ARABIC LETTER DDAL ISOLATED FORM
        '\uFB89',  # ARABIC LETTER DDAL FINAL FORM
    ],
    # ZAL (ذ)
    '\u0630': [
        '\u0630',  # ARABIC LETTER THAL
        '\uFEAB',  # ARABIC LETTER THAL ISOLATED FORM
        '\uFEAC',  # ARABIC LETTER THAL FINAL FORM
    ],
    # RE (ر)
    '\u0631': [
        '\u0631',  # ARABIC LETTER REH
        '\uFEAD',  # ARABIC LETTER REH ISOLATED FORM
        '\uFEAE',  # ARABIC LETTER REH FINAL FORM
    ],
    # RRE (ڑ)
    '\u0691': [
        '\u0691',  # ARABIC LETTER RREH
        '\uFB8C',  # ARABIC LETTER RREH ISOLATED FORM
        '\uFB8D',  # ARABIC LETTER RREH FINAL FORM
    ],
    # ZE (ز)
    '\u0632': [
        '\u0632',  # ARABIC LETTER ZAIN
        '\uFEAF',  # ARABIC LETTER ZAIN ISOLATED FORM
        '\uFEB0',  # ARABIC LETTER ZAIN FINAL FORM
    ],
    # ZHE (ژ)
    '\u0698': [
        '\u0698',  # ARABIC LETTER JEH
        '\uFB8A',  # ARABIC LETTER JEH ISOLATED FORM
        '\uFB8B',  # ARABIC LETTER JEH FINAL FORM
    ],
    # SEEN (س)
    '\u0633': [
        '\u0633',  # ARABIC LETTER SEEN
        '\uFEB1',  # ARABIC LETTER SEEN ISOLATED FORM
        '\uFEB2',  # ARABIC LETTER SEEN FINAL FORM
        '\uFEB3',  # ARABIC LETTER SEEN INITIAL FORM
        '\uFEB4',  # ARABIC LETTER SEEN MEDIAL FORM
    ],
    # SHEEN (ش)
    '\u0634': [
        '\u0634',  # ARABIC LETTER SHEEN
        '\uFEB5',  # ARABIC LETTER SHEEN ISOLATED FORM
        '\uFEB6',  # ARABIC LETTER SHEEN FINAL FORM
        '\uFEB7',  # ARABIC LETTER SHEEN INITIAL FORM
        '\uFEB8',  # ARABIC LETTER SHEEN MEDIAL FORM
    ],
    # SUAD (ص)
    '\u0635': [
        '\u0635',  # ARABIC LETTER SAD
        '\uFEB9',  # ARABIC LETTER SAD ISOLATED FORM
        '\uFEBA',  # ARABIC LETTER SAD FINAL FORM
        '\uFEBB',  # ARABIC LETTER SAD INITIAL FORM
        '\uFEBC',  # ARABIC LETTER SAD MEDIAL FORM
    ],
    # ZUAD (ض)
    '\u0636': [
        '\u0636',  # ARABIC LETTER DAD
        '\uFEBD',  # ARABIC LETTER DAD ISOLATED FORM
        '\uFEBE',  # ARABIC LETTER DAD FINAL FORM
        '\uFEBF',  # ARABIC LETTER DAD INITIAL FORM
        '\uFEC0',  # ARABIC LETTER DAD MEDIAL FORM
    ],
    # TO'E (ط)
    '\u0637': [
        '\u0637',  # ARABIC LETTER TAH
        '\uFEC1',  # ARABIC LETTER TAH ISOLATED FORM
        '\uFEC2',  # ARABIC LETTER TAH FINAL FORM
        '\uFEC3',  # ARABIC LETTER TAH INITIAL FORM
        '\uFEC4',  # ARABIC LETTER TAH MEDIAL FORM
    ],
    # ZO'E (ظ)
    '\u0638': [
        '\u0638',  # ARABIC LETTER ZAH
        '\uFEC5',  # ARABIC LETTER ZAH ISOLATED FORM
        '\uFEC6',  # ARABIC LETTER ZAH FINAL FORM
        '\uFEC7',  # ARABIC LETTER ZAH INITIAL FORM
        '\uFEC8',  # ARABIC LETTER ZAH MEDIAL FORM
    ],
    # AIN (ع)
    '\u0639': [
        '\u0639',  # ARABIC LETTER AIN
        '\uFEC9',  # ARABIC LETTER AIN ISOLATED FORM
        '\uFECA',  # ARABIC LETTER AIN FINAL FORM
        '\uFECB',  # ARABIC LETTER AIN INITIAL FORM
        '\uFECC',  # ARABIC LETTER AIN MEDIAL FORM
    ],
    # GHAIN (غ)
    '\u063A': [
        '\u063A',  # ARABIC LETTER GHAIN
        '\uFECD',  # ARABIC LETTER GHAIN ISOLATED FORM
        '\uFECE',  # ARABIC LETTER GHAIN FINAL FORM
        '\uFECF',  # ARABIC LETTER GHAIN INITIAL FORM
        '\uFED0',  # ARABIC LETTER GHAIN MEDIAL FORM
    ],
    # FE (ف)
    '\u0641': [
        '\u0641',  # ARABIC LETTER FE
        '\uFED1',  # ARABIC LETTER FE ISOLATED FORM
        '\uFED2',  # ARABIC LETTER FE FINAL FORM
        '\uFED3',  # ARABIC LETTER FE INITIAL FORM
        '\uFED4',  # ARABIC LETTER FE MEDIAL FORM
    ],
    # QAF (ق)
    '\u0642': [
        '\u0642',  # ARABIC LETTER QAF
        '\uFED5',  # ARABIC LETTER QAF ISOLATED FORM
        '\uFED6',  # ARABIC LETTER QAF FINAL FORM
        '\uFED7',  # ARABIC LETTER QAF INITIAL FORM
        '\uFED8',  # ARABIC LETTER QAF MEDIAL FORM
    ],
    # URDU KAF (ک)
    '\u06A9': [
        # Urdu/Persian Keheh (ک)
        '\u06A9',  # ARABIC LETTER KEHEH
        '\uFB8E',  # ARABIC LETTER KEHEH ISOLATED FORM
        '\uFB8F',  # ARABIC LETTER KEHEH FINAL FORM
        '\uFB90',  # ARABIC LETTER KEHEH INITIAL FORM
        '\uFB91',  # ARABIC LETTER KEHEH MEDIAL FORM
        # Arabic Kaf (ك)
        '\u0643',  # ARABIC LETTER KAF
        '\uFED9',  # ARABIC LETTER KAF ISOLATED FORM
        '\uFEDA',  # ARABIC LETTER KAF FINAL FORM
        '\uFEDB',  # ARABIC LETTER KAF INITIAL FORM
        '\uFEDC',  # ARABIC LETTER KAF MEDIAL FORM
        # Old Persian / Swash Kaf (ݢ)
        '\u06A8',  # ARABIC LETTER KAF WITH TWO DOTS ABOVE
        '\uFB96',  # ARABIC LETTER KAF WITH TWO DOTS ABOVE ISOLATED FORM
        '\uFB97',  # ARABIC LETTER KAF WITH TWO DOTS ABOVE FINAL FORM
        '\uFB98',  # ARABIC LETTER KAF WITH TWO DOTS ABOVE INITIAL FORM
        '\uFB99',  # ARABIC LETTER KAF WITH TWO DOTS ABOVE MEDIAL FORM
    ],
    # GAF (گ)
    '\u06AF': [
        '\u06AF',  # ARABIC LETTER GAF
        '\uFB92',  # ARABIC LETTER GAF ISOLATED FORM
        '\uFB93',  # ARABIC LETTER GAF FINAL FORM
        '\uFB94',  # ARABIC LETTER GAF INITIAL FORM
        '\uFB95',  # ARABIC LETTER GAF MEDIAL FORM
    ],
    # LAM (ل)
    '\u0644': [
        '\u0644',  # ARABIC LETTER LAM
        '\uFEDD',  # ARABIC LETTER LAM ISOLATED FORM
        '\uFEDE',  # ARABIC LETTER LAM FINAL FORM
        '\uFEDF',  # ARABIC LETTER LAM INITIAL FORM
        '\uFEE0',  # ARABIC LETTER LAM MEDIAL FORM
    ],
    # MEEM (م)
    '\u0645': [
        '\u0645',  # ARABIC LETTER MEEM
        '\uFEE1',  # ARABIC LETTER MEEM ISOLATED FORM
        '\uFEE2',  # ARABIC LETTER MEEM FINAL FORM
        '\uFEE3',  # ARABIC LETTER MEEM INITIAL FORM
        '\uFEE4',  # ARABIC LETTER MEEM MEDIAL FORM
    ],
    # NOON (ن)
    '\u0646': [
        '\u0646',  # ARABIC LETTER NOON
        '\uFEE5',  # ARABIC LETTER NOON ISOLATED FORM
        '\uFEE6',  # ARABIC LETTER NOON FINAL FORM
        '\uFEE7',  # ARABIC LETTER NOON INITIAL FORM
        '\uFEE8',  # ARABIC LETTER NOON MEDIAL FORM
    ],
    # NOON GHUNNA (ں)
    '\u06BA': [
        '\u06BA',  # ARABIC LETTER NOON GHUNNA
        '\uFB9E',  # ARABIC LETTER NOON GHUNNA ISOLATED FORM
        '\uFB9F',  # ARABIC LETTER NOON GHUNNA FINAL FORM
    ],
    # WAO (و)
    '\u0648': [
        # Standard Wao
        '\u0648',  # ARABIC LETTER WAW
        '\uFEED',  # ARABIC LETTER WAW ISOLATED FORM
        '\uFEEE',  # ARABIC LETTER WAW FINAL FORM
        # Wao with Hamza (ؤ)
        '\u0624',  # ARABIC LETTER WAW WITH HAMZA ABOVE
        '\uFE85',  # ARABIC LETTER WAW WITH HAMZA ABOVE ISOLATED FORM
        '\uFE86',  # ARABIC LETTER WAW WITH HAMZA ABOVE FINAL FORM
    ],
    # HE GOAL (ہ)
    '\u06C1': [
        # Urdu Heh Goal (ہ)
        '\u06C1',  # ARABIC LETTER HEH GOAL
        '\uFBA6',  # ARABIC LETTER HEH GOAL ISOLATED FORM
        '\uFBA7',  # ARABIC LETTER HEH GOAL FINAL FORM
        '\uFBA8',  # ARABIC LETTER HEH GOAL INITIAL FORM
        '\uFBA9',  # ARABIC LETTER HEH GOAL MEDIAL FORM
        # Arabic Ha (ه)
        '\u0647',  # ARABIC LETTER HEH
        '\uFEE9',  # ARABIC LETTER HEH ISOLATED FORM
        '\uFEEA',  # ARABIC LETTER HEH FINAL FORM
        '\uFEEB',  # ARABIC LETTER HEH INITIAL FORM
        '\uFEEC',  # ARABIC LETTER HEH MEDIAL FORM
        # Heh with Hamza variants (ۂ)
        '\u06C2',  # ARABIC LETTER HEH GOAL WITH HAMZA ABOVE
        '\u06C0',  # ARABIC LETTER HEH WITH YEH ABOVE
        # Te Marbuta (ة)
        '\u0629',  # ARABIC LETTER TE MARBUTA
        '\uFE93',  # ARABIC LETTER TE MARBUTA ISOLATED FORM
        '\uFE94',  # ARABIC LETTER TE MARBUTA FINAL FORM
    ],
    # DO CHASHMI HE (ھ)
    '\u06BE': [
        '\u06BE',  # ARABIC LETTER HEH DOACHASHMEE
        '\uFBAC',  # ARABIC LETTER HEH DOACHASHMEE ISOLATED FORM
        '\uFBAD',  # ARABIC LETTER HEH DOACHASHMEE FINAL FORM
        '\uFBAE',  # ARABIC LETTER HEH DOACHASHMEE INITIAL FORM
        '\uFBAF',  # ARABIC LETTER HEH DOACHASHMEE MEDIAL FORM
    ],
    # CHOTI YE / FARSI YEH / ARABIC YEH (ی / ي)
    '\u06CC': [
        # Urdu/Farsi Standard (ی)
        '\u06CC',  # ARABIC LETTER FARSI YEH
        '\uFBFB',  # ARABIC LETTER FARSI YEH ISOLATED FORM
        '\uFBFC',  # ARABIC LETTER FARSI YEH FINAL FORM
        '\uFBFD',  # ARABIC LETTER FARSI YEH INITIAL FORM
        '\uFBFE',  # ARABIC LETTER FARSI YEH MEDIAL FORM
        # Arabic/Sindhi Standard (ي)
        '\u064A',  # ARABIC LETTER YEH
        '\uFEF1',  # ARABIC LETTER YEH ISOLATED FORM
        '\uFEF2',  # ARABIC LETTER YEH FINAL FORM
        '\uFEF3',  # ARABIC LETTER YEH INITIAL FORM
        '\uFEF4',  # ARABIC LETTER YEH MEDIAL FORM
        # Alef Maksura (ى)
        '\u0649',  # ARABIC LETTER ALEF MAKSURA
        '\uEEF1',  # ARABIC LETTER ALEF MAKSURA ISOLATED FORM
        '\uEEF2',  # ARABIC LETTER ALEF MAKSURA FINAL FORM
    ],
    '\u0626': [ # Yeh with Hamza (ئ) - Common compositional variant
        '\u0626',  # ARABIC LETTER YEH WITH HAMZA ABOVE
        '\uFE8B',  # ARABIC LETTER YEH WITH HAMZA ABOVE INITIAL FORM
        '\uFE8C',  # ARABIC LETTER YEH WITH HAMZA ABOVE MEDIAL FORM
        '\uFE89',  # ARABIC LETTER YEH WITH HAMZA ABOVE ISOLATED FORM
        '\uFE8A',  # ARABIC LETTER YEH WITH HAMZA ABOVE FINAL FORM
    ],
    # BARI YE (ے)
    '\u06D2': [
        '\u06D2',  # ARABIC LETTER YEH BARREE
        '\uFBAE',  # ARABIC LETTER YEH BARREE ISOLATED FORM
        '\uFBAF',  # ARABIC LETTER YEH BARREE FINAL FORM
        '\u06D3',  # ARABIC LETTER YEH BARREE WITH HAMZA ABOVE
    ],
    # KASHEEDA / TATWEEL (ـ)
    # Mapping to empty string is a common way to 'strip' it during normalization
    '': [
        '\u0640',  # ARABIC TATWEEL
    ],
}

LIGATURE_MAP = {
    '\uFEFB': '\u0644\u0627',  # LAM WITH ALEF ISOLATED -> ل + ا
    '\uFEFC': '\u0644\u0627',  # LAM WITH ALEF FINAL -> ل + ا
    '\uFEF5': '\u0644\u0622',  # LAM WITH ALEF MADDA ISOLATED -> ل + آ
    '\uFEF6': '\u0644\u0622',  # LAM WITH ALEF MADDA FINAL -> ل + آ
}

URDU_REVERSAL_LOOKUP = {v: base for base, variants in URDU_VARIANT_MAP.items() for v in variants}

def normalizeUrduChars(text):
    """
    Replaces presentation/positional forms with standard Urdu tokens
    using the built-in map function for performance.
    """
    if not text:
        return ""

    # Step 1: Handle Multi-character Ligatures
    # We use a regex for efficiency if the ligature list grows,
    # but for a small set, a simple loop or multiple .replace() works well.
    for ligature, replacement in LIGATURE_MAP.items():
        text = text.replace(ligature, replacement)

    # Step 2: Handle 1-to-1 Character Variants using map()
    # This cleans up positional forms (initial, medial, etc.)
    # map() applies the lambda to every character in the string.
    # .get(c, c) ensures we keep characters not in our dictionary (like spaces/punctuation).
    text = "".join(map(lambda c: URDU_REVERSAL_LOOKUP.get(c, c), text))

    return text


In [13]:
testToken = 'ﺗﻨﻈﻴﻤﻮں'
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalizeUrduChars(testToken)
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ﺗﻨﻈﻴﻤﻮں [U+06BA U+FEEE U+FEE4 U+FEF4 U+FEC8 U+FEE8 U+FE97]
Normalized: تنظیموں [U+06BA U+0648 U+0645 U+06CC U+0638 U+0646 U+062A]


In [5]:
def removeMarks(text):
    # Selective Diacritic Removal (NFD)
    # NFD is used to isolate marks, then filter them out.
    # Marks to preserve:
    # * U+0653 (Madda - for آ)
    # * U+0654 (Hamza Above - for ئ),
    # other base characters are filtered by character normalization
    preservedMarks = {'\u0653', '\u0654'}
    nfdForm = ud.normalize('NFD', text)
    token = "".join([c for c in nfdForm if c in preservedMarks or not ud.combining(c)])
    return ud.normalize('NFC', token)

def removePunctuation(text):
    return "".join([c for c in text if not ud.category(c).startswith('P')])

def removeDigits(text):
    return "".join([c for c in text if not ud.category(c) == 'Nd'])

def normalizeNonChars(text):
    return removePunctuation(removeDigits(removeMarks(text)))

def normalizeWhiteSpace(text):
    return  " ".join(text.split())

def normalize(text):
    return normalizeWhiteSpace(normalizeNonChars(normalizeUrduChars(text)))


In [28]:
testToken = 'ـ'
print(f"Original: {testToken} [{getUniRep(testToken)}]")
tt = normalizeWhiteSpace(normalizeNonChars(testToken))
print(f"Normalized: {tt} [{getUniRep(tt)}]")

Original: ـ [U+0640]
Normalized: ـ [U+0640]


In [9]:
def get5klines(filename):
    """Yields batches of lines from a file."""
    with open(filename, 'r', encoding='utf8') as file:
        batch = []
        for line in file:
            batch.append(line)
            if len(batch) == 5000:
                yield batch
                batch = []

        # Yield any remaining lines if the file isn't a perfect multiple of 5000
        if batch:
            yield batch

In [46]:
def generateVocab(filename):
    nlp = spacy.blank('ur')
    i = 1
    vocab = Counter()
    for lines in get5klines(filename):
        batch = []
        for line in lines:
            doc = nlp(line)
            words = [w for w in [normalize(token.text) for token in doc if scriptScore(token.text) > 0.0] if w]
            batch.extend(words)
        vocab.update(Counter(batch))
        print(f"Vocab[{i:>2}]: {len(vocab)}")
        i += 1
    return vocab

In [22]:
def saveVocab(filename, wordCounter, sep='$'):
    with open(filename, "w", encoding="utf-8") as dic:
        for i, (w, f) in enumerate(wordCounter.most_common()):
            dic.write(f"{w}{sep}{f}\n")

In [20]:
def loadVocabCounter(filename, sep='$'):
    """Loads word frequency pairs into a Counter object."""
    counter = Counter()
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Split by the specific '$' delimiter
                word, freq = line.rsplit(sep, 1)
                counter[word] = int(freq)
    return counter

In [47]:
files = [
    'wordlist.txt',
    'training_Urdu_PK.txt',
    'Firdaus-e-Bareen.txt',
    'Inkithab-e-Meer.txt',
    'ParlimanyMujray.txt',
    'RajaGidh.txt'
]

mainVocab = Counter()
for f in files:
    vv = generateVocab(f)
    mainVocab.update(vv)

saveVocab("UrduVocab.sym", mainVocab)

Vocab[ 1]: 4497
Vocab[ 2]: 9140
Vocab[ 3]: 13366
Vocab[ 4]: 17479
Vocab[ 5]: 21678
Vocab[ 6]: 26030
Vocab[ 7]: 29810
Vocab[ 8]: 33658
Vocab[ 9]: 37405
Vocab[10]: 41191
Vocab[11]: 44911
Vocab[12]: 47652
Vocab[13]: 50610
Vocab[14]: 54003
Vocab[15]: 57031
Vocab[16]: 60589
Vocab[17]: 63464
Vocab[18]: 66124
Vocab[19]: 69239
Vocab[20]: 72401
Vocab[21]: 75759
Vocab[22]: 79098
Vocab[23]: 81687
Vocab[24]: 84229
Vocab[25]: 86452
Vocab[26]: 88876
Vocab[27]: 90739
Vocab[28]: 94103
Vocab[29]: 97439
Vocab[30]: 100261
Vocab[ 1]: 4160
h LATIN
t LATIN
t LATIN
p LATIN
u LATIN
r LATIN
d LATIN
u LATIN
l LATIN
i LATIN
b LATIN
r LATIN
a LATIN
r LATIN
y LATIN
o LATIN
r LATIN
g LATIN
h LATIN
t LATIN
t LATIN
p LATIN
k LATIN
i LATIN
t LATIN
a LATIN
b LATIN
e LATIN
n LATIN
i LATIN
f LATIN
a LATIN
s LATIN
t LATIN
n LATIN
e LATIN
t LATIN
c LATIN
o LATIN
m LATIN
h LATIN
t LATIN
t LATIN
p LATIN
k LATIN
u LATIN
t LATIN
u LATIN
b LATIN
f LATIN
r LATIN
e LATIN
e LATIN
c LATIN
o LATIN
m LATIN
Vocab[ 1]: 6741
Vocab[ 1]: 

In [58]:
class Vocabulary():
    def __init__(self, referenceVocabulary=None, sep='$'):
        self.symSpell = SymSpell()
        if referenceVocabulary:
            self.loadReference(referenceVocabulary, sep)
        self.vocab = Counter()

    def loadReference(self, filename, sep='$'):
        self.symSpell.load_dictionary(filename, 0, 1, separator=sep, encoding='utf8')

    def exists(self, word):
        """Check if word exists in reference vocabulary"""
        suggestions = self.symSpell.lookup(word, Verbosity.CLOSEST, max_edit_distance=0)
        return len(suggestions) > 0

    def suggest(self, word, distance=1):
        """Get suggestions from reference vocabulary"""
        return self.symSpell.lookup(word, Verbosity.CLOSEST, max_edit_distance=distance)

    def extract(self, text, filterKnown=True):
        """Extract word-frequency pairs from given text"""
        nlp = spacy.blank('ur')
        doc = nlp(text)
        # Normalize all potential tokens
        tokens = [normalize(token.text) for token in doc if scriptScore(token.text) > 0.0]
        print(f"{len(tokens)=}")
        for t in tokens:
            print(f"token: {t} [{getUniRep(t)}]")
        # Optionally filter against the reference
        words = [w for w in tokens if w and (filterKnown and not self.exists(w))]
        print(f"{len(words)=}")
        print(f"Before {len(self.vocab)=}")
        self.vocab.update(Counter(words))
        print(f"After {len(self.vocab)=}")

    @property
    def words(self):
        return [w for w, f in self.vocab.most_common()]

    def save(self, filename, sep='$'):
        """Saves word-frequency pairs to a SymSpell file"""
        with open(filename, "w", encoding="utf-8") as sym:
            for i, (w, f) in enumerate(self.vocab.most_common()):
                sym.write(f"{w}{sep}{f}\n")

    def load(self, filename, sep='$'):
        """Loads word-frequency pairs from a SymSpell file"""
        counter = Counter()
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    # Split by the specific '$' delimiter
                    word, freq = line.rsplit(sep, 1)
                    counter[word] = int(freq)
        self.vocab = counter

In [59]:
v = Vocabulary("UrduVocab.sym")

In [60]:
v.exists('\u0627\u0628')

True

In [65]:
txt = Path('fiaz.txt').read_text(encoding='utf-8-sig')

In [66]:
v.extract(txt)

len(tokens)=16
token: اب [U+0628 U+0627]
token: کے [U+06D2 U+06A9]
token: ہم [U+0645 U+06C1]
token: بچھڑے [U+06D2 U+0691 U+06BE U+0686 U+0628]
token: تو [U+0648 U+062A]
token: شائد [U+062F U+0626 U+0627 U+0634]
token: خوابوں [U+06BA U+0648 U+0628 U+0627 U+0648 U+062E]
token: ملیں [U+06BA U+06CC U+0644 U+0645]
token:  []
token: جیسے [U+06D2 U+0633 U+06CC U+062C]
token: سوکھے [U+06D2 U+06BE U+06A9 U+0648 U+0633]
token: ہوئے [U+06D2 U+0626 U+0648 U+06C1]
token: پھول [U+0644 U+0648 U+06BE U+067E]
token: کتابوں [U+06BA U+0648 U+0628 U+0627 U+062A U+06A9]
token: میں [U+06BA U+06CC U+0645]
token: ملیں [U+06BA U+06CC U+0644 U+0645]
len(words)=1
Before len(self.vocab)=2
After len(self.vocab)=2
